In [35]:
import pandas as pd

# Read all sheets from our input file
params   = pd.read_excel('TEA_Input_Tables.xlsx', sheet_name='01_Global_Parameters', skiprows=2)
equip    = pd.read_excel('TEA_Input_Tables.xlsx', sheet_name='02_Equipment_List',    skiprows=2)
opex_df  = pd.read_excel('TEA_Input_Tables.xlsx', sheet_name='03_OPEX_Input',        skiprows=2)
factors  = pd.read_excel('TEA_Input_Tables.xlsx', sheet_name='04_CAPEX_Factors',     skiprows=2)
cepci_df = pd.read_excel('TEA_Input_Tables.xlsx', sheet_name='05_CEPCI_Table',       skiprows=2)

print("Excel file read successfully!")
print(f"Equipment items: {len(equip)}")
print(f"OPEX items: {len(opex_df)}")
print(f"CAPEX factors: {len(factors)}")
print()
print("Equipment list:")
print(equip[['Equipment Name', 'Type', 'S_new (from Aspen)', 'Quantity']])

Excel file read successfully!
Equipment items: 25
OPEX items: 8
CAPEX factors: 14

Equipment list:
   Equipment Name            Type  S_new (from Aspen)  Quantity
0    Compressor 1      Compressor           11500.000         1
1    Compressor 2      Compressor           11160.000         1
2    Compressor 3      Compressor           11370.000         1
3    Compressor 4      Compressor           12580.000         1
4    Compressor 5      Compressor             570.000         1
5            Pump            Pump               0.044         1
6    Flash drum 1          Vessel             124.000         1
7    Flash drum 2          Vessel              46.000         1
8    Flash drum 3          Vessel             165.000         1
9    Flash drum 4          Vessel             116.000         1
10           HX-1  Heat Exchanger            5800.000         1
11           HX-2  Heat Exchanger            3610.000         1
12           HX-3  Heat Exchanger            1160.000         1
13   

In [36]:
# Extract global parameters from the table
# We convert the Parameter column into a dictionary for easy access
param_dict = dict(zip(params['Parameter'], params['Value']))

# Now we can access each parameter by name
CEPCI_current  = param_dict['CEPCI_current']
CEPCI_1998     = param_dict['CEPCI_1998']
CEPCI_2002     = param_dict['CEPCI_2002']
USD_to_EUR     = param_dict['USD_to_EUR']
discount_rate  = param_dict['discount_rate']
tax_rate       = param_dict['tax_rate']
plant_lifetime = int(param_dict['plant_lifetime'])
operating_hours= int(param_dict['operating_hours'])
methanol_price = param_dict['methanol_price']

print("Global parameters loaded from Excel!")
print(f"  CEPCI current : {CEPCI_current}")
print(f"  Discount rate : {discount_rate*100}%")
print(f"  Plant lifetime: {plant_lifetime} years")

Global parameters loaded from Excel!
  CEPCI current : 596.2
  Discount rate : 10.0%
  Plant lifetime: 20 years


In [37]:
# ============================================================
# COMPLETE CAPEX CALCULATION - FROM EQUIPMENT TO FCI
# Run this cell FIRST to make sure FCI is always available
# ============================================================

import pandas as pd

# --- Global Parameters ---
CEPCI_current  = 596.2
USD_to_EUR     = 1.13      # multiply (matches thesis Excel convention)
discount_rate  = 0.10
tax_rate       = 0.00
plant_lifetime = 20
operating_hours = 8000
methanol_price  = 920

# --- CEPCI lookup ---
cepci_lookup = dict(zip(cepci_df['Year'], cepci_df['CEPCI']))

# --- Reactor volume correction (validated value) ---
equip.loc[equip['Equipment Name'] == 'Reactor', 'S_new (from Aspen)'] = 432.4
equip.loc[equip['Equipment Name'] == 'Reactor', 'C_ref (USD)']        = 85878
equip.loc[equip['Equipment Name'] == 'Reactor', 'S_ref']              = 0.80
equip['f_material'] = equip['f_material'].astype(float)
equip.loc[equip['Equipment Name'] == 'Reactor', 'f_material']         = 1.04

# --- Equipment cost calculation (TPEC) ---
results = []
for _, row in equip.iterrows():
    cepci_ref   = cepci_lookup[row['CEPCI_ref_year']]
    C_scaled    = row['C_ref (USD)'] * (row['S_new (from Aspen)'] / row['S_ref']) ** row['Exponent']
    C_updated   = C_scaled * (CEPCI_current / cepci_ref)
    C_purchase  = C_updated * row['f_material'] * USD_to_EUR
    C_total     = C_purchase * row['Quantity']
    C_installed = C_total * row['f_install']
    results.append({
        'Equipment':           row['Equipment Name'],
        'Purchase Cost (M€)':  round(C_total / 1e6, 3),
        'Installed Cost (M€)': round(C_installed / 1e6, 3),
    })

df_capex = pd.DataFrame(results)
TPEC = df_capex['Purchase Cost (M€)'].sum()
TPEC_EUR = TPEC * 1e6

# --- FCI calculation (factorial method) ---
direct_costs   = (0.55+0.10+0.18+0.11+0.68+0.36+0.47) * TPEC_EUR
indirect_costs = (0.33+0.41+0.04+0.2115+0.423) * TPEC_EUR

FCI = TPEC_EUR + direct_costs + indirect_costs
WCI = 0.5405 * TPEC_EUR
TCI = FCI + WCI

print("=" * 55)
print("CAPEX SUMMARY (FCI now available for OPEX)")
print("=" * 55)
print(f"TPEC: {TPEC_EUR/1e6:.2f} M€  ← Thesis: 86.3 M€")
print(f"FCI:  {FCI/1e6:.2f} M€  ← Thesis: 419.9 M€")
print(f"WCI:  {WCI/1e6:.2f} M€  ← Thesis: 46.6 M€")
print(f"TCI:  {TCI/1e6:.2f} M€  ← Thesis: 466.6 M€")
print("=" * 55)

CAPEX SUMMARY (FCI now available for OPEX)
TPEC: 85.23 M€  ← Thesis: 86.3 M€
FCI:  414.61 M€  ← Thesis: 419.9 M€
WCI:  46.07 M€  ← Thesis: 46.6 M€
TCI:  460.67 M€  ← Thesis: 466.6 M€


In [38]:
# ============================================================
# OPEX CALCULATION - DIRECT AND INDIRECT
# Source: Thesis Table 12, page 37
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# PART 1: DIRECT OPEX (Variable Costs)
# ------------------------------------------------------------
voc_results = []
for _, row in opex_df.iterrows():
    if row['Category'] == 'Labour':
        continue
    annual_cost = row['Annual Quantity'] * row['Price (EUR)']
    voc_results.append({
        'Item':             row['Item'],
        'Annual Cost (M€)': round(annual_cost / 1e6, 3)
    })

df_direct = pd.DataFrame(voc_results)
Direct_OPEX = df_direct['Annual Cost (M€)'].sum()

# Thesis Direct OPEX values from Table 12, page 37
thesis_direct = {
    'Hydrogen H2':    770.50,
    'CO2':             74.93,
    'Cooling water':    0.38,
    'Clean water':      0.02,
    'TOC disposal':     0.12,
    'Electricity':     12.66,
    'Catalyst':        17.31,
}

df_direct['Thesis (M€)'] = df_direct['Item'].map(thesis_direct)
df_direct['Diff (M€)']   = df_direct['Annual Cost (M€)'] - df_direct['Thesis (M€)']

print("=" * 60)
print("DIRECT OPEX (Variable Costs)")
print("=" * 60)
print(df_direct.to_string(index=False))
print("-" * 60)
print(f"Total Direct OPEX (Python): {Direct_OPEX:.2f} M€")
print(f"Total Direct OPEX (Thesis): 874.94 M€")
print(f"Difference: {(Direct_OPEX - 874.94):.2f} M€  ({(Direct_OPEX-874.94)/874.94*100:.2f}%)")
print()

# ------------------------------------------------------------
# PART 2: INDIRECT OPEX (Fixed Costs)
# Using FCI from CAPEX calculation
# ------------------------------------------------------------
OL = 970000           # Operating Labour from Excel input (EUR/year)
OS = 0.15 * OL         # Operating Supervision = 15% OL

# Maintenance = 2% of FCI (Towler & Sinnott standard)
maintenance = 0.02 * FCI
ML = maintenance / 2
MM = maintenance / 2

supplies    = 0.15 * (ML + MM)
lab_charges = 0.20 * OL
insurance   = 0.02 * FCI
overhead    = 0.60 * (OL + OS + ML)
admin       = 0.25 * overhead

# Distribution and R&D depend on NPC (circular - solved by iteration)
FOC_base = OL + OS + ML + MM + supplies + lab_charges + insurance + overhead + admin

CRF = (discount_rate * (1+discount_rate)**plant_lifetime) / \
      ((1+discount_rate)**plant_lifetime - 1)
ACC = FCI * CRF

NPC_iter = Direct_OPEX*1e6 + FOC_base + ACC
for _ in range(5):
    distribution = 0.06 * NPC_iter
    RD           = 0.04 * NPC_iter
    FOC_total    = FOC_base + distribution + RD
    OPEX_total   = Direct_OPEX*1e6 + FOC_total
    NPC_iter     = OPEX_total + ACC

# Build indirect OPEX table
indirect_items = [
    ('Operating Labour',       OL),
    ('Operating Supervision',  OS),
    ('Maintenance Labour',     ML),
    ('Maintenance Material',   MM),
    ('Operating Supplies',     supplies),
    ('Laboratory Charges',     lab_charges),
    ('Insurance and Taxes',    insurance),
    ('Plant Overhead',         overhead),
    ('Administrative Costs',   admin),
    ('Distribution & Marketing', distribution),
    ('Research & Development', RD),
]

# Thesis Indirect OPEX values from Table 12, page 37
thesis_indirect = {
    'Operating Labour':         1.15,
    'Operating Supervision':    0.17,
    'Maintenance Labour':       8.06,
    'Maintenance Material':     8.06,
    'Operating Supplies':       2.42,
    'Laboratory Charges':       0.23,
    'Insurance and Taxes':      8.06,
    'Plant Overhead':           5.63,
    'Administrative Costs':     1.41,
    'Distribution & Marketing': 64.13,
    'Research & Development':   42.75,
}

df_indirect = pd.DataFrame(indirect_items, columns=['Item', 'Python (EUR)'])
df_indirect['Python (M€)'] = (df_indirect['Python (EUR)'] / 1e6).round(3)
df_indirect['Thesis (M€)'] = df_indirect['Item'].map(thesis_indirect)
df_indirect['Diff (M€)']   = df_indirect['Python (M€)'] - df_indirect['Thesis (M€)']
df_indirect = df_indirect[['Item', 'Python (M€)', 'Thesis (M€)', 'Diff (M€)']]

print("=" * 60)
print("INDIRECT OPEX (Fixed Costs)")
print("=" * 60)
print(df_indirect.to_string(index=False))
print("-" * 60)
print(f"Total Indirect OPEX (Python): {FOC_total/1e6:.2f} M€")
print(f"Total Indirect OPEX (Thesis): 143.40 M€")
print(f"Difference: {(FOC_total/1e6 - 143.40):.2f} M€  ({(FOC_total/1e6-143.40)/143.40*100:.2f}%)")
print()

# ------------------------------------------------------------
# PART 3: TOTAL OPEX
# ------------------------------------------------------------
print("=" * 60)
print("TOTAL OPEX SUMMARY")
print("=" * 60)
print(f"Direct OPEX:   {Direct_OPEX:>8.2f} M€  ← Thesis: 874.94 M€")
print(f"Indirect OPEX: {FOC_total/1e6:>8.2f} M€  ← Thesis: 143.40 M€")
print(f"Total OPEX:    {OPEX_total/1e6:>8.2f} M€  ← Thesis: 1018.30 M€")
print("=" * 60)

DIRECT OPEX (Variable Costs)
         Item  Annual Cost (M€)  Thesis (M€)  Diff (M€)
  Hydrogen H2           770.633       770.50      0.133
          CO2            74.956        74.93      0.026
     Catalyst            16.942        17.31     -0.368
  Electricity            12.663        12.66      0.003
  Clean water             0.023         0.02      0.003
Cooling water             0.378         0.38     -0.002
 TOC disposal             0.120         0.12      0.000
------------------------------------------------------------
Total Direct OPEX (Python): 875.72 M€
Total Direct OPEX (Thesis): 874.94 M€
Difference: 0.78 M€  (0.09%)

INDIRECT OPEX (Fixed Costs)
                    Item  Python (M€)  Thesis (M€)  Diff (M€)
        Operating Labour        0.970         1.15     -0.180
   Operating Supervision        0.146         0.17     -0.024
      Maintenance Labour        4.146         8.06     -3.914
    Maintenance Material        4.146         8.06     -3.914
      Operating Su